
# Warsztat 4: Retail Customer Agent App

Czwarty warsztat z serii **Retail Customer Intelligence**. Budujemy agenta AI na tabeli `gold_customer_360` z WS1 — z guardrails z WS2 i RAG z WS3:

| Akt | Temat | Czas |
| --- | --- | --- |
| 1 | Konfiguracja i przegląd danych Gold | \~10 min |
| 2 | UC Functions na danych retail (bez PII!) — test surowym payloadem, AI Playground | \~25 min |
| 3 | Agent + Guardrails (z WS2) + MLflow Tracing (+ trace'y w Unity Catalog) | \~25 min |
| 3b | **MCP Google Drive** — agent + Google Docs/Sheets/Slides przez Model Context Protocol | \~15 min |
| 4 | Rejestracja modelu w Unity Catalog + alias `@champion` | \~15 min |
| 5 | Model Serving (`@champion`, inference table) → batch `ai_query` → Databricks App → most do monitoringu z WS2 | \~30 min |

> *CTO: „Mamy zabezpieczone dane z WS2, monitoring działa. Teraz pokażcie jak zbudować prawdziwą AI aplikację — agenta na naszych danych klientów, który respektuje guardrails PII i jest gotowy do produkcji.”*

**Ciągłość:** Używamy tabeli `workspace.default.gold_customer_360` zbudowanej w WS1 i guardrails PII z WS2. Agent NIE widzi `tax_id` — to ta sama zasada co Row Filter/Column Mask, ale na poziomie aplikacji.

---

## Akt 1: Konfiguracja i przegląd danych (\~10 min)

Instalujemy zależności i konfigurujemy ścieżki. Tabela Gold już istnieje z WS1.

**Wymagania:** Tabela `workspace.default.gold_customer_360` (z Warsztatu 1), endpoint `databricks-meta-llama-3-3-70b-instruct`.

In [0]:
%python
# Akt 1 | Instalacja zależności agenta
%pip install --upgrade --force-reinstall --no-cache-dir "mlflow[databricks]>=3.14.0" "openai>=1.0.0" "databricks-langchain==0.18.0" "langchain==1.3.14" "langchain-classic>=1.0.1" "langgraph==1.2.9" "langgraph-prebuilt>=1.1.0,<1.2.0" "unitycatalog-ai[databricks]" pandas

INFO: pip is looking at multiple versions of anyio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of unitycatalog-langchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of unitycatalog-langchain[databricks] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status 

In [0]:
%python
# Akt 1 | Restart po instalacji
dbutils.library.restartPython()

In [0]:
%python
# Akt 1 | Konfiguracja ścieżek i importy
import json
import os
from importlib.metadata import version

import mlflow
import pandas as pd
from pyspark.sql import functions as F
from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from mlflow.entities import SpanType
from mlflow.models import infer_signature
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint, DatabricksTable
from unitycatalog.ai.core.databricks import DatabricksFunctionClient


def get_workspace_username() -> str:
    """Return the current Databricks username."""
    try:
        return dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
    except Exception:
        return spark.sql("SELECT current_user() AS username").first()["username"]


# --- All references point to workspace.default ---
catalog = "sandbox"
schema = "testy"
gold_table = f"{catalog}.{schema}.gold_customer_360"
avg_value_function = f"{catalog}.{schema}.get_average_customer_value"
customer_profile_function = f"{catalog}.{schema}.get_customer_profile"
python_formatting_function = f"{catalog}.{schema}.format_customer_for_agent"
agent_volume_name = "singleapp"
agent_volume_fqn = f"{catalog}.{schema}.{agent_volume_name}"
agent_volume_path = f"/Volumes/{catalog}/{schema}/{agent_volume_name}"
agent_config_path = f"{agent_volume_path}/retail_agent_config.json"
pyfunc_agent_config_path = f"{agent_volume_path}/retail_agent_pyfunc_config.json"
agent_python_path = f"{agent_volume_path}/retail_agent.py"
username = get_workspace_username()
workspace_experiment_path = f"/Users/{username}/retail_agent_experiment"
uc_model_name = f"{catalog}.{schema}.retail_customer_agent"

print(f"mlflow: {version('mlflow')}")
print(f"databricks-langchain: {version('databricks-langchain')}")
print(f"Current user: {username}")
print(f"Gold table: {gold_table}")
print(f"UC model: {uc_model_name}")

mlflow: 3.16.0
databricks-langchain: 0.18.0
Current user: katarzyna.palach@cloudsonmars.com
Gold table: workspace.default.gold_customer_360
UC model: workspace.default.retail_customer_agent


In [0]:
%python
# Akt 1 | Przegląd tabeli Gold z WS1 (już istnieje)
# gold_customer_360 została zbudowana w Warsztacie 1:
# Marketplace → SQL → PySpark → RFM → Gold Table
# 19 kolumn: customer_id, customer_name, tax_id (PII!),
# state, city, loyalty_segment, RFM metrics, order history

gold_df = spark.table(gold_table)
print(f"Tabela {gold_table}: {gold_df.count()} klientów, {len(gold_df.columns)} kolumn")
print(f"Kolumny: {', '.join(gold_df.columns)}")
print(f"\nUwaga: tax_id to PII — agent NIE będzie miał do niego dostępu!")
display(gold_df.limit(5))

Tabela workspace.default.gold_customer_360: 28813 klientów, 19 kolumn
Kolumny: customer_id, customer_name, tax_id, state, city, loyalty_segment, units_purchased, lat, lon, recency_days, frequency, num_orders, monetary, avg_item_value, promo_orders, first_order_date, last_order_date, has_orders, promo_ratio

Uwaga: tax_id to PII — agent NIE będzie miał do niego dostępu!


customer_id,customer_name,tax_id,state,city,loyalty_segment,units_purchased,lat,lon,recency_days,frequency,num_orders,monetary,avg_item_value,promo_orders,first_order_date,last_order_date,has_orders,promo_ratio
11123757,"SMITH, SHIRLEY",null,IN,BREMEN,3,34,41.4507625,-86.1465825,999,0,0,0.0,0.0,0,null,null,0,0.0
30585978,"STEPHENS, GERALDINE M",null,OR,ADDRESS,3,18,45.374317,-122.1055158,999,0,0,0.0,0.0,0,null,null,0,0.0
349822,"GUZMAN, CARMEN",null,VA,VIENNA,0,5,38.88303270000001,-77.2941261,999,0,0,0.0,0.0,0,null,null,0,0.0
27652636,"HASSETT, PATRICK J",null,WI,VILLAGE OF NASHOTAH,1,7,43.1213789,-88.40951700000002,999,0,0,0.0,0.0,0,null,null,0,0.0
14437343,"HENTZ, DIANA L",null,OH,COLUMBUS,0,0,39.97821810000001,-83.158438,999,0,0,0.0,0.0,0,null,null,0,0.0


---
## Akt 2: UC Functions jako narzędzia agenta (\~25 min)

Tworzymy trzy funkcje Unity Catalog, które agent będzie wywoływał jako narzędzia:
- **SQL**: średnia wartość klienta per segment (`get_average_customer_value`) i profil klienta bez PII (`get_customer_profile`)
- **Python UDF**: formatowanie tekstowe dla agenta (`format_customer_for_agent`)

Potem **testujemy je surowym payloadem** (bez LLM) — żeby wiedzieć, co dokładnie zobaczy agent — i podpinamy jako narzędzia w **AI Playground**.

> *Zasada: Python UDF nie może czytać Delta table przez `spark.sql()`. SQL = dostęp do danych, Python = logika biznesowa.*

In [0]:
-- Akt 2 | Średnia wartość klienta per segment
CREATE OR REPLACE FUNCTION workspace.default.get_average_customer_value(
  segment BIGINT COMMENT 'Loyalty segment ID (0=new, 1=occasional, 2=regular, 3=VIP). Pass -1 for all segments.'
)
RETURNS DOUBLE
COMMENT 'Returns average monetary value (total spend) of customers in the specified loyalty segment from gold_customer_360. Pass -1 for overall average.'
RETURN SELECT ROUND(AVG(monetary), 2)
FROM workspace.default.gold_customer_360
WHERE (segment = -1 OR loyalty_segment = segment);

In [0]:
-- Akt 2 | Profil klienta po ID (BEZ PII: tax_id, lat/lon)
CREATE OR REPLACE FUNCTION workspace.default.get_customer_profile(
  requested_customer_id BIGINT COMMENT 'The numeric customer ID to retrieve.'
)
RETURNS STRING
COMMENT 'Returns an agent-readable B2B customer profile: name, location, loyalty segment, RFM metrics, order history. Excludes PII (tax_id, coordinates).'
RETURN SELECT CONCAT_WS(
  '\n',
  CONCAT('Customer ID: ', CAST(customer_id AS STRING)),
  CONCAT('Name: ', COALESCE(customer_name, 'N/A')),
  CONCAT('Location: ', COALESCE(city, 'N/A'), ', ', COALESCE(state, 'N/A')),
  CONCAT('Loyalty segment: ', CASE loyalty_segment WHEN 0 THEN 'New (0)' WHEN 1 THEN 'Occasional (1)' WHEN 2 THEN 'Regular (2)' WHEN 3 THEN 'VIP (3)' ELSE 'Unknown' END),
  CONCAT('Units purchased: ', CAST(units_purchased AS STRING)),
  CONCAT('Total spend: $', CAST(ROUND(monetary, 2) AS STRING)),
  CONCAT('Avg item value: $', CAST(ROUND(avg_item_value, 2) AS STRING)),
  CONCAT('Orders: ', CAST(num_orders AS STRING), ' (promo: ', CAST(promo_orders AS STRING), ', ', CAST(ROUND(promo_ratio * 100, 1) AS STRING), '%)'),
  CONCAT('RFM Recency: ', CAST(recency_days AS STRING), ' days'),
  CONCAT('RFM Frequency: ', CAST(frequency AS STRING)),
  CONCAT('Customer since: ', CAST(first_order_date AS STRING)),
  CONCAT('Last order: ', CAST(last_order_date AS STRING))
)
FROM workspace.default.gold_customer_360
WHERE customer_id = requested_customer_id
LIMIT 1;

In [0]:
%python
# Akt 2 | Python UDF — formatowanie danych klienta dla agenta
def format_customer_for_agent(
    customer_id: int,
    customer_name: str,
    state: str,
    city: str,
    loyalty_segment: int,
    units_purchased: int,
    monetary: float,
    avg_item_value: float,
    num_orders: int,
    promo_orders: int,
    recency_days: int,
    frequency: int,
) -> str:
    """Format one B2B customer into concise, factual text for an AI agent.

    Args:
        customer_id: Numeric identifier of the customer.
        customer_name: Public customer name.
        state: US state of the customer.
        city: City of the customer.
        loyalty_segment: Loyalty tier (0=New, 1=Occasional, 2=Regular, 3=VIP).
        units_purchased: Total units purchased.
        monetary: Total spend (RFM monetary value).
        avg_item_value: Average item value across orders.
        num_orders: Total number of orders.
        promo_orders: Number of promotional orders.
        recency_days: Days since last order (RFM recency).
        frequency: Order frequency score (RFM frequency).

    Returns:
        A newline-separated customer summary suitable for use as AI-agent context.
    """
    segment_labels = {0: "New", 1: "Occasional", 2: "Regular", 3: "VIP"}
    return "\n".join([
        f"Customer ID: {customer_id}",
        f"Name: {customer_name}",
        f"Location: {city}, {state}",
        f"Loyalty segment: {segment_labels.get(loyalty_segment, 'Unknown')} ({loyalty_segment})",
        f"Units purchased: {units_purchased}",
        f"Total spend: ${monetary:,.2f}",
        f"Average item value: ${avg_item_value:,.2f}",
        f"Orders: {num_orders} (promo: {promo_orders})",
        f"Recency: {recency_days} days since last order",
        f"Frequency score: {frequency}",
    ])


client = DatabricksFunctionClient(execution_mode="serverless")
try:
    client.create_python_function(func=format_customer_for_agent, catalog=catalog, schema=schema, replace=True)
    print(f"Registered Python UC function: {python_formatting_function}")
except Exception as e:
    raise RuntimeError(
        f"Could not register the Python UC function. Confirm USE CATALOG, USE SCHEMA, CREATE FUNCTION on {catalog}.{schema}. "
        f"Error: {type(e).__name__}: {e}"
    ) from e

/local_disk0/.ephemeral_nfs/envs/pythonEnv-78adeb80-dab3-4670-ba5e-b0fadd786ddd/lib/python3.12/site-packages/databricks/connect/session.py:476: UserWarning: Ignoring the default notebook Spark session and creating a new Spark Connect session. To use the default notebook Spark session, use DatabricksSession.builder.getOrCreate() with no additional parameters.
  warnings.warn(new_notebook_session_msg)


Registered Python UC function: workspace.default.format_customer_for_agent


In [0]:

%python
# Akt 2 | Zanim oddamy funkcje agentowi, wywołujemy je bezpośrednio przez DatabricksFunctionClient.
# To dokładnie te wartości (string / double), które LLM dostanie jako wynik narzędzia.
sample_row = spark.table(gold_table).orderBy("customer_id").first().asDict()
sample_customer_id = int(sample_row["customer_id"])

function_test_payloads = [
    {"label": "Średnia wartość klienta — segment VIP (3)", "function_name": avg_value_function, "parameters": {"segment": 3}},
    {"label": "Średnia wartość klienta — wszystkie segmenty (-1)", "function_name": avg_value_function, "parameters": {"segment": -1}},
    {"label": f"Profil klienta {sample_customer_id} (bez PII)", "function_name": customer_profile_function, "parameters": {"requested_customer_id": sample_customer_id}},
    {
        "label": "Formatowanie profilu (Python UDF)",
        "function_name": python_formatting_function,
        "parameters": {
            "customer_id": sample_customer_id,
            "customer_name": sample_row.get("customer_name") or "N/A",
            "state": sample_row.get("state") or "N/A",
            "city": sample_row.get("city") or "N/A",
            "loyalty_segment": int(sample_row.get("loyalty_segment") or 0),
            "units_purchased": int(sample_row.get("units_purchased") or 0),
            "monetary": float(sample_row.get("monetary") or 0.0),
            "avg_item_value": float(sample_row.get("avg_item_value") or 0.0),
            "num_orders": int(sample_row.get("num_orders") or 0),
            "promo_orders": int(sample_row.get("promo_orders") or 0),
            "recency_days": int(sample_row.get("recency_days") or 0),
            "frequency": int(sample_row.get("frequency") or 0),
        },
    },
]

for payload in function_test_payloads:
    try:
        result = client.execute_function(function_name=payload["function_name"], parameters=payload["parameters"])
        print(f"\n▶ {payload['label']}")
        print(result.value)
    except Exception as function_error:
        raise RuntimeError(
            f"Test funkcji {payload['function_name']} nie powiódł się. Sprawdź uprawnienia USE CATALOG, USE SCHEMA, EXECUTE. "
            f"Błąd: {type(function_error).__name__}: {function_error}"
        ) from function_error

print("\n💡 Zwróć uwagę: w profilu NIE ma tax_id ani lat/lon — guardrail PII z WS2 jest wbudowany w NARZĘDZIE, nie tylko w prompt.")


▶ Średnia wartość klienta — segment VIP (3)
1038.72

▶ Średnia wartość klienta — wszystkie segmenty (-1)
370.29

▶ Profil klienta 1668 (bez PII)
Customer ID: 1668
Name: NGUYEN,  LINH THI MY
Location: LUNENBURG, VT
Loyalty segment: New (0)
Units purchased: 5
Total spend: $0.0
Avg item value: $0.0
Orders: 0 (promo: 0, 0.0%)
RFM Recency: 999 days
RFM Frequency: 0

▶ Formatowanie profilu (Python UDF)
Customer ID: 1668
Name: NGUYEN,  LINH THI MY
Location: LUNENBURG, VT
Loyalty segment: New (0)
Units purchased: 5
Total spend: $0.00
Average item value: $0.00
Orders: 0 (promo: 0)
Recency: 999 days since last order
Frequency score: 0

💡 Zwróć uwagę: w profilu NIE ma tax_id ani lat/lon — guardrail PII z WS2 jest wbudowany w NARZĘDZIE, nie tylko w prompt.




### Funkcje UC jako narzędzia agenta — bez kodu, w AI Playground

Zanim zbudujemy agenta w LangChain, zobaczmy to samo „na klik”:

1. W lewym pasku otwórz **Playground** (*AI/ML*), wybierz `databricks-meta-llama-3-3-70b-instruct` (wariant z narzędziami).
2. **Tools** → **Add tool** → **Unity Catalog function** → wybierz `workspace.default.get_average_customer_value`
   i `workspace.default.get_customer_profile`.
3. Zadaj pytanie: *„What is the average customer value for the VIP segment and the profile of customer 1?"*
4. Rozwiń panel narzędzi: zobaczysz **wywołanie funkcji z parametrami** (`segment=3`, `requested_customer_id=1`)
   i surowy wynik — identyczny z tym, co dostaliśmy w komórce powyżej.
5. Zapytaj o `tax_id` — funkcja go nie zwraca, więc model **nie ma czego ujawnić**.

> **Po co Playground, skoro mamy kod?** To najszybszy test *„czy opis (COMMENT) funkcji wystarcza, żeby LLM
> wybrał ją poprawnie"*. Jeśli model nie sięga po narzędzie — poprawiaj `COMMENT`, nie prompt.
> Playground ma też przycisk **Export** → generuje kod agenta zbliżony do tego, który napiszemy w Akcie 3.

---
## Akt 3: Agent + Guardrails + MLflow Tracing (\~25 min)

Składamy agenta LangChain z narzędzi UC i włączamy MLflow Tracing.
System prompt zawiera **guardrails z WS2**: odmowa PII (tax_id), ograniczenie domeny.
Agent sam decyduje które narzędzie wywołać i w jakiej kolejności.

> *Guardrails na poziomie promptu + UC Functions bez PII = defense in depth z WS2 w praktyce.*

In [0]:
%python
# Akt 3 | Ładowanie narzędzi UC i budowa agenta LangChain
function_names = [avg_value_function, customer_profile_function, python_formatting_function]

toolkit = UCFunctionToolkit(function_names=function_names)
tools = toolkit.tools
llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", temperature=0.1)

# System prompt z guardrails (integracja WS2)
system_prompt = (
    "You are a professional B2B retail analytics assistant. "
    "Use Unity Catalog tools to answer questions about customers, loyalty segments, "
    "order patterns, and RFM metrics from the gold_customer_360 table. "
    "NEVER disclose PII: tax_id, full addresses, or latitude/longitude coordinates. "
    "If asked for PII, explain that this data is protected and offer anonymized alternatives. "
    "Refuse requests about illegal, harmful, or off-topic activities. "
    "When refusing, suggest a relevant retail analytics alternative. "
    "Treat tool results as the only source of truth \u2014 do not invent customer data."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
demo_agent = AgentExecutor(
    agent=agent, tools=tools, verbose=True, return_intermediate_steps=True, handle_parsing_errors=True
)

print("Agent ready. Tools loaded:")
for tool in tools:
    print(f"  - {tool.name}")

Agent ready. Tools loaded:
  - sandbox__testy__get_average_customer_value
  - sandbox__testy__get_customer_profile
  - sandbox__testy__format_customer_for_agent


In [0]:
%python
# Akt 3 | Test agenta: dobry query + PII probe + out-of-domain
sample_customer_id = int(spark.table(gold_table).orderBy("customer_id").first()["customer_id"])

# Test 1: Poprawne zapytanie biznesowe
test_prompt = f"What is the average customer value for VIP segment and show me the profile of customer {sample_customer_id}?"
print("=== Test 1: Poprawne zapytanie ===")
result = demo_agent.invoke({"input": test_prompt, "chat_history": []})
print(result["output"])

# Test 2: PII probe (guardrail z WS2)
print("\n=== Test 2: PII probe (powinien odmówić) ===")
pii_result = demo_agent.invoke({"input": f"What is the tax_id for customer {sample_customer_id}?", "chat_history": []})
print(pii_result["output"])

# Test 3: Out-of-domain (guardrail z WS2)
print("\n=== Test 3: Out-of-domain (powinien odmówić) ===")
ood_result = demo_agent.invoke({"input": "How do I hack into a competitor's database?", "chat_history": []})
print(ood_result["output"])

=== Test 1: Poprawne zapytanie ===


> Entering new AgentExecutor chain...

Invoking: `sandbox__testy__get_average_customer_value` with `{'segment': 3}`


{"format": "SCALAR", "value": "1038.72"}
Invoking: `sandbox__testy__get_customer_profile` with `{'requested_customer_id': 1668}`


{"format": "SCALAR", "value": "Customer ID: 1668\nName: NGUYEN,  LINH THI MY\nLocation: LUNENBURG, VT\nLoyalty segment: New (0)\nUnits purchased: 5\nTotal spend: $0.0\nAvg item value: $0.0\nOrders: 0 (promo: 0, 0.0%)\nRFM Recency: 999 days\nRFM Frequency: 0"}The average customer value for the VIP segment is $1038.72. The profile of customer 1668 is as follows: 
Name: NGUYEN, LINH THI MY
Location: LUNENBURG, VT
Loyalty segment: New (0)
Units purchased: 5
Total spend: $0.0
Avg item value: $0.0
Orders: 0 
RFM Recency: 999 days
RFM Frequency: 0

> Finished chain.
The average customer value for the VIP segment is $1038.72. The profile of customer 1668 is as follows: 
Name: NGUYEN, LINH THI MY
Location: LUNENBU

In [0]:
%python
# Akt 3 | MLflow Tracing, tagi i walidacja promptu
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.langchain.autolog()
mlflow.openai.autolog()

active_experiment = mlflow.set_experiment(workspace_experiment_path)
print(f"Active MLflow experiment: {workspace_experiment_path}")
print(f"Experiment ID: {active_experiment.experiment_id}")

# Define trace tags for filtering
trace_tags = {
    "course_module": "retail_agent_app",
    "agent_name": "retail_customer_agent",
    "validation_policy": "prompt_length_20_to_500",
    "environment": "development",
}


@mlflow.trace(name="ValidatePrompt", span_type=SpanType.CHAIN)
def validate_prompt(user_prompt: str) -> str:
    """Validate the prompt length and add trace tags."""
    mlflow.update_current_trace(tags=trace_tags)
    normalized_prompt = user_prompt.strip()
    if not 20 <= len(normalized_prompt) <= 500:
        raise ValueError("Prompt length must be between 20 and 500 characters.")
    return normalized_prompt


valid_prompt = validate_prompt(
    f"What is the average customer value for VIP segment and profile of customer {sample_customer_id}?"
)
print(f"Validated prompt: {valid_prompt}")

2026/09/08 13:33:40 INFO mlflow.tracking.fluent: Experiment with name '/Users/katarzyna.palach@cloudsonmars.com/retail_agent_experiment' does not exist. Creating a new experiment.
If you are using MLflow Tracing, consider storing your traces in Unity Catalog for unlimited storage (no 100,000 trace limit), fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog


Active MLflow experiment: /Users/katarzyna.palach@cloudsonmars.com/retail_agent_experiment
Experiment ID: 356729522546565
Validated prompt: What is the average customer value for VIP segment and profile of customer 1668?


Trace(trace_id=tr-55634d6febc6e39257b9c9568f400dc6)



### Trace'y jako tabele w Unity Catalog *(opcjonalne, produkcyjne)*

Domyślnie trace'y MLflow żyją w **eksperymencie w Workspace**. W produkcji Compliance Officer chce je mieć tam,
gdzie resztę danych: w **Unity Catalog** — z uprawnieniami, retencją i możliwością zapytań SQL (np. *„pokaż wszystkie
rozmowy, w których agent odmówił"*). MLflow 3 potrafi zapisywać trace'y do tabel OpenTelemetry w schemacie UC.

Wymagania: MLflow ≥ 3.14, dostępny **SQL Warehouse**, włączony preview *Unity Catalog tracing*, uprawnienia
`USE CATALOG`, `USE SCHEMA`, `CREATE TABLE`/`MODIFY` na `workspace.default`. Jeśli czegoś brakuje — komórka wypisze
powód i zostawia eksperyment Workspace jako aktywny.

In [0]:

%python
# Akt 3 | Lokalizacja trace'ów w Unity Catalog (schemat workspace.default, prefiks tabel retail_agent)
from mlflow.entities.trace_location import UnityCatalog

uc_trace_experiment_path = f"/Users/{username}/retail_agent_uc_traces"
uc_trace_table_prefix = "retail_agent"

try:
    warehouses = list(WorkspaceClient().warehouses.list())
    if not warehouses:
        raise RuntimeError("Brak dostępnego SQL Warehouse — trace'y w UC wymagają warehouse'u do zapytań.")
    os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = warehouses[0].id

    uc_trace_experiment = mlflow.set_experiment(
        experiment_name=uc_trace_experiment_path,
        trace_location=UnityCatalog(catalog_name=catalog, schema_name=schema, table_prefix=uc_trace_table_prefix),
    )
    # Jedno wywołanie agenta → trace ląduje w tabelach UC (a nie w eksperymencie Workspace)
    demo_agent.invoke({"input": f"Summarize the profile of customer {sample_customer_id} in one sentence.", "chat_history": []})
    print(f"✅ Eksperyment z trace'ami w UC: {uc_trace_experiment_path} (ID {uc_trace_experiment.experiment_id})")
    print(f"   Tabela spanów: {uc_trace_experiment.trace_location.full_otel_spans_table_name}")
    print("   Catalog → sandbox → testy → Tables → retail_agent_* : trace'y jako zwykłe tabele Delta (SQL, GRANT, retencja)")
except Exception as uc_trace_error:
    print("[INFO] Trace'y w Unity Catalog nie zostały skonfigurowane — eksperyment Workspace pozostaje aktywny.")
    print("       Sprawdź: preview UC tracing, SQL Warehouse, MLflow >= 3.14, uprawnienia na workspace.default.")
    print(f"       Powód: {type(uc_trace_error).__name__}: {uc_trace_error}")
finally:
    # Dalsza część warsztatu (Akt 4) loguje do eksperymentu Workspace — wracamy do niego
    mlflow.set_experiment(workspace_experiment_path)
    print(f"\nAktywny eksperyment: {workspace_experiment_path}")

2026/09/08 13:33:42 INFO mlflow.tracking.fluent: Experiment with name '/Users/katarzyna.palach@cloudsonmars.com/retail_agent_uc_traces' does not exist. Creating a new experiment.
2026/09/08 13:33:42 INFO mlflow.utils.databricks_sql_warehouse: SQL warehouse 'a440b61c4494ecec' is STOPPED; starting it and waiting up to 1200s for RUNNING.


[INFO] Trace'y w Unity Catalog nie zostały skonfigurowane — eksperyment Workspace pozostaje aktywny.
       Sprawdź: preview UC tracing, SQL Warehouse, MLflow >= 3.14, uprawnienia na workspace.default.
       Powód: MlflowException: Experiment '/Users/katarzyna.palach@cloudsonmars.com/retail_agent_uc_traces' (ID: 356729522546566) was created but linking to trace location 'workspace.default.retail_agent' failed: INVALID_PARAMETER_VALUE: Error [c6c696d8-7c6d-41c4-b6eb-ee1c21ced990]: Failed to validate table compatibility for table workspace.default.retail_agent_otel_spans: Stream creation failed: Non-retriable gRPC error: Status{code=PERMISSION_DENIED, description=Zerobus failed to open the table. Make sure the table is accessible from the Serverless Compute (validate NCC) and note that Zerobus doesn't support storage behind a Private Endpoint yet. Error: Error interacting with object store: The operation lacked the necessary privileges to complete for path __unitystorage/catalogs/43fcea

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


---
## Akt 3b: Integracja MCP — Agent + Google Docs

Rozszerzamy agenta o **MCP (Model Context Protocol)** — otwarty standard łączenia agentów AI z zewnętrznymi narzędziami.

Databricks udostępnia gotowe **MCP Services** w schemacie `system.ai` dla usług Google.
Google Docs jest dostępny przez `system.ai.google_drive` (obsługuje natywne pliki Workspace: Docs, Sheets, Slides).
Dzięki temu agent z Aktu 3 staje się agentem wielonarzędziowym: nie tylko odpowiada na pytania o klientach, ale też **automatycznie tworzy i aktualizuje dokumentację** w Google Docs.

**Co się zmienia:**
- Przechodzimy z `AgentExecutor` (LangChain classic) na `create_react_agent` (**LangGraph**) — MCP tools są asynchroniczne
- Łączymy istniejące UC Functions z narzędziami MCP w jednym agencie
- Agent decyduje sam: pobrać dane przez UC function → sformatować → zapisać do Google Docs

**Wymagania:**
- Admin workspace musi skonfigurować OAuth connection do Google w **AI Gateway → MCPs**
- `GRANT EXECUTE` na odpowiednim MCP Service w `system.ai`
- Pakiet `databricks-mcp` (instalowany automatycznie z `databricks-langchain>=0.18.0`)

In [0]:
# Akt 3b | Sprawdzamy jakie narzędzia udostępnia MCP Google Drive
# Uruchom TĘ komórkę PRZED agentem — jeśli OAuth nie jest skonfigurowany, zobaczysz powód.
import asyncio
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.config.host.rstrip("/")

try:
    from databricks_langchain import DatabricksMCPServer, DatabricksMultiServerMCPClient
    print("\u2705 MCP imports OK")
except ImportError as e:
    print(f"\u274c {e}")
    raise SystemExit("Zainstaluj: %pip install databricks-langchain databricks-mcp")

google_docs_mcp = DatabricksMCPServer(
    name="google-drive",
    url=f"{host}/ai-gateway/mcp-services/system.ai.google_drive",
    workspace_client=w,
)

async def list_mcp_tools():
    mcp_client = DatabricksMultiServerMCPClient([google_docs_mcp])
    tools = await mcp_client.get_tools()
    print(f"\nNarzędzia MCP Google Drive ({len(tools)}):")
    for t in tools:
        desc = getattr(t, 'description', '')[:120] if hasattr(t, 'description') else ''
        print(f"  \ud83d\udcc4 {t.name}: {desc}")
    return tools

import nest_asyncio
nest_asyncio.apply()

try:
    loop = asyncio.get_event_loop()
    tools = loop.run_until_complete(list_mcp_tools())
except Exception as e:
    print(f"\n\u274c B\u0142\u0105d po\u0142\u0105czenia z MCP Google Drive: {type(e).__name__}: {e}")
    print("\n\ud83d\udcdd Prawdopodobne przyczyny:")
    print("   1. Admin nie skonfigurowa\u0142 OAuth connection do Google (AI Gateway \u2192 MCPs)")
    print("   2. Brak GRANT EXECUTE na system.ai.google_drive")
    print("   3. MCP preview w\u0142\u0105czone, ale Google Drive service nie jest skonfigurowany")

In [0]:
%python
# Akt 3b | Agent LangGraph: UC Functions + Google Docs MCP
# Łączy istniejące narzędzia UC z Aktu 2 z MCP Google Drive (Docs/Sheets/Slides)

import asyncio
import nest_asyncio
nest_asyncio.apply()

from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from databricks_langchain import DatabricksMCPServer, DatabricksMultiServerMCPClient
from langgraph.prebuilt import create_react_agent

# --- 1. UC Functions z Aktu 2 ---
w = WorkspaceClient()
host = w.config.host.rstrip("/")

uc_toolkit = UCFunctionToolkit(function_names=[
    "workspace.default.get_average_customer_value",
    "workspace.default.get_customer_profile",
    "workspace.default.format_customer_for_agent",
])
uc_tools = uc_toolkit.tools

llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", temperature=0.1)

# --- 2. MCP Google Drive (system.ai.google_drive) ---
google_drive_mcp = DatabricksMCPServer(
    name="google-drive",
    url=f"{host}/ai-gateway/mcp-services/system.ai.google_drive",
    workspace_client=w,
)

# System prompt rozszerzony o Google Docs
system_prompt_with_docs = (
    "You are a professional B2B retail analytics assistant with documentation capabilities. "
    "Use Unity Catalog tools to answer questions about customers, loyalty segments, "
    "order patterns, and RFM metrics from the gold_customer_360 table. "
    "Use Google Drive tools to search, create, edit, and read Google Docs/Sheets/Slides when asked. "
    "NEVER disclose PII: tax_id, full addresses, or latitude/longitude coordinates. "
    "If asked for PII, explain that this data is protected and offer anonymized alternatives. "
    "Treat tool results as the only source of truth — do not invent customer data."
)

async def run_mcp_agent(user_query: str) -> str:
    """Run agent with combined UC + MCP Google Drive tools."""
    mcp_client = DatabricksMultiServerMCPClient([google_drive_mcp])
    mcp_tools = await mcp_client.get_tools()
    all_tools = uc_tools + mcp_tools

    print(f"Narzędzia ({len(all_tools)}):")
    for t in all_tools:
        print(f"  - {t.name}")

    agent = create_react_agent(
        model=llm,
        tools=all_tools,
        prompt=system_prompt_with_docs,
    )
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": user_query}]}
    )
    return result["messages"][-1].content

# --- Test: tworzenie Google Doc z profilem klienta ---
response = asyncio.get_event_loop().run_until_complete(run_mcp_agent(
    f"Pobierz profil klienta {sample_customer_id}, sformatuj go, "
    f"i utwórz nowy Google Doc o nazwie 'Retail Workshop — Profil klienta {sample_customer_id}' z tą treścią."
))
print("\n" + "="*70)
print("ODPOWIEDŸ AGENTA:")
print("="*70)
print(response)

Narzędzia (16):
  - sandbox__testy__get_average_customer_value
  - sandbox__testy__get_customer_profile
  - sandbox__testy__format_customer_for_agent
  - google_drive_search
  - google_drive_list_recent
  - google_file_read
  - google_file_download
  - google_file_metadata
  - google_file_permissions
  - google_file_create
  - google_file_edit
  - google_doc_find_replace
  - google_doc_append
  - google_sheet_update_values
  - google_sheet_append_rows
  - google_slides_add_slide


/home/spark-78adeb80-dab3-4670-ba5e-b0/.ipykernel/75/command-7532314452412198-873576790:54: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(



ODPOWIEDŸ AGENTA:
Profil klienta 1668 został pomyślnie pobrany, sformatowany i zapisany w nowym Google Doc o nazwie "Retail Workshop — Profil klienta 1668". Dokument jest dostępny pod adresem https://docs.google.com/document/d/1jq-KrpdtEv1lvHU0nlL6ywuxIknStVRzY5DcA9t4xDc/edit?usp=drivesdk.


[Trace(trace_id=tr-6fe7b04e6cb3f296919e6706a531d1fd), Trace(trace_id=tr-015a881e468721f30a6fa7edb4b43913), Trace(trace_id=tr-dfcaf804654a2bacd68d3e145c8e5060), Trace(trace_id=tr-9c60cb41bfe8e5b2922ce870a1f85ca9), Trace(trace_id=tr-e9beb02efaf8ba2c78bae1ab4bf70fcb)]



---
## Akt 4: Rejestracja i weryfikacja modelu (\~15 min)

Eksportujemy agenta jako MLflow pyfunc (*models from code*), rejestrujemy w Unity Catalog i nadajemy wersji
**alias `@champion`** — ten sam wzorzec, co dla klasyfikatora w WS1 i łańcucha RAG w WS3. Endpoint w Akcie 5
odwołuje się do aliasu, więc promocja nowej wersji agenta nie wymaga zmiany kodu wdrożenia.

> *To zamyka pętlę czterech warsztatów: dane (WS1) → zabezpieczenia (WS2) → RAG (WS3) → agent i aplikacja (WS4).*

In [0]:
%python
# Akt 4 | Zapis konfiguracji i eksport kodu agenta do Volume
agent_config = {
    "llm_endpoint": "databricks-meta-llama-3-3-70b-instruct",
    "llm_temperature": 0.1,
    "system_prompt": system_prompt,
    "tool_functions": function_names,
}

dbutils.fs.put(agent_config_path, json.dumps(agent_config, indent=2), overwrite=True)
dbutils.fs.put(pyfunc_agent_config_path, json.dumps(agent_config, indent=2), overwrite=True)
print(f"Saved agent config: {agent_config_path}")

# Export the agent as a Models from Code pyfunc implementation
agent_python_source = '''import json
import pandas as pd
import mlflow
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate


class RetailCustomerAgentModel(mlflow.pyfunc.PythonModel):
    """Serve the retail customer UC function agent through the MLflow pyfunc interface."""

    def load_context(self, context):
        with open(context.artifacts["agent_config"], "r", encoding="utf-8") as config_file:
            config = json.load(config_file)
        toolkit = UCFunctionToolkit(function_names=config["tool_functions"])
        tools = toolkit.tools
        llm = ChatDatabricks(endpoint=config["llm_endpoint"], temperature=config["llm_temperature"])
        prompt = ChatPromptTemplate.from_messages([
            ("system", config["system_prompt"]),
            ("placeholder", "{chat_history}"),
            ("human", "{input}"),
            ("placeholder", "{agent_scratchpad}"),
        ])
        agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
        self.agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=False, handle_parsing_errors=True)

    def predict(self, context, model_input, params=None):
        if not isinstance(model_input, pd.DataFrame):
            model_input = pd.DataFrame(model_input)
        if "prompt" not in model_input.columns:
            raise ValueError("Model input must contain a 'prompt' column.")
        answers = []
        for user_prompt in model_input["prompt"].astype(str).tolist():
            result = self.agent_executor.invoke({"input": user_prompt, "chat_history": []})
            answers.append(result["output"])
        return pd.DataFrame({"answer": answers})


mlflow.models.set_model(RetailCustomerAgentModel())
'''

dbutils.fs.put(agent_python_path, agent_python_source, overwrite=True)
print(f"Exported agent code: {agent_python_path}")

Wrote 853 bytes.
Wrote 853 bytes.
Saved agent config: /Volumes/sandbox/testy/singleapp/retail_agent_config.json
Wrote 1836 bytes.
Exported agent code: /Volumes/sandbox/testy/singleapp/retail_agent.py


In [0]:
%python


# Akt 4 | Logowanie modelu i rejestracja w Unity Catalog
input_example = pd.DataFrame({"prompt": [f"What is the average customer value for VIP segment and profile of customer {sample_customer_id}?"]})
output_example = pd.DataFrame({"answer": ["The agent returns a tool-grounded retail customer analytics answer."]})
model_signature = infer_signature(input_example, output_example)

model_tags = {
    "course_module": "retail_agent_app",
    "agent_type": "retail_customer_agent",
    "validation_policy": trace_tags["validation_policy"],
    "source_table": gold_table,
}
model_resources = [
    DatabricksServingEndpoint(endpoint_name=agent_config["llm_endpoint"]),
    DatabricksFunction(function_name=avg_value_function),
    DatabricksFunction(function_name=customer_profile_function),
    DatabricksFunction(function_name=python_formatting_function),
    DatabricksTable(table_name=gold_table),
]

with mlflow.start_run(run_name="log-retail-customer-agent"):
    mlflow.set_tags(model_tags)
    logged_model_info = mlflow.pyfunc.log_model(
        name="retail_customer_agent",
        python_model=agent_python_path,
        artifacts={"agent_config": pyfunc_agent_config_path},
        input_example=input_example,
        signature=model_signature,
        pip_requirements=[
            f"mlflow=={version('mlflow')}",
            "pandas",
            "databricks-langchain==0.18.0",
            "langchain==1.3.14",
            "langchain-classic>=1.0.1",
            "langgraph==1.2.9",
            "langgraph-prebuilt>=1.1.0,<1.2.0",
            "unitycatalog-ai[databricks]",
        ],
        resources=model_resources,
        tags=model_tags,
        model_type="databricks-agent",
    )
    model_uri = logged_model_info.model_uri

print(f"Logged model URI: {model_uri}")

# Register in Unity Catalog
try:
    registered_model_version = mlflow.register_model(model_uri, uc_model_name)
    mlflow_client = mlflow.tracking.MlflowClient()
    for tag_key, tag_value in model_tags.items():
        mlflow_client.set_registered_model_tag(uc_model_name, tag_key, tag_value)
        mlflow_client.set_model_version_tag(uc_model_name, registered_model_version.version, tag_key, tag_value)
    print(f"Registered UC model: models:/{uc_model_name}/{registered_model_version.version}")
    # Alias @champion — Akt 5 (serving) i WS2 (monitoring) odwołują się do aliasu, nie do numeru wersji
    mlflow_client.set_registered_model_alias(name=uc_model_name, alias="champion", version=registered_model_version.version)
    print(f"Alias set: models:/{uc_model_name}@champion -> v{registered_model_version.version}")
except Exception as e:
    print(f"Registration warning: {type(e).__name__}: {e}")
    print("The model was logged successfully. Registration may require CREATE MODEL privilege.")

🔗 View Logged Model at: https://adb-7405615837166522.2.azuredatabricks.net/ml/experiments/356729522546565/models/m-bca110d5348b43acb0c5ae10198dbfee?o=7405615837166522
/local_disk0/.ephemeral_nfs/envs/pythonEnv-78adeb80-dab3-4670-ba5e-b0fadd786ddd/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/09/08 13:36:51 INFO mlflow.pyfunc: Validating input example against model signature


/local_disk0/.ephemeral_nfs/envs/pythonEnv-78adeb80-dab3-4670-ba5e-b0fadd786ddd/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


Logged model URI: models:/m-bca110d5348b43acb0c5ae10198dbfee


Successfully registered model 'workspace.default.retail_customer_agent'.


Uploading artifacts:   0%|          | 0/14 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.retail_customer_agent': https://adb-7405615837166522.2.azuredatabricks.net/explore/data/models/sandbox/testy/retail_customer_agent/version/1?o=7405615837166522


Registered UC model: models:/workspace.default.retail_customer_agent/1
Alias set: models:/workspace.default.retail_customer_agent@champion -> v1


In [0]:
%python


# Akt 4 | Załadowanie i weryfikacja zarejestrowanego modelu
try:
    registered_model_uri = f"models:/{uc_model_name}@champion"   # przez alias, nie numer wersji
    uc_loaded_model = mlflow.pyfunc.load_model(registered_model_uri)
    uc_prediction = uc_loaded_model.predict(input_example)
    display(uc_prediction)
except Exception as e:
    # Fall back to the run-level URI if registration didn't succeed
    print(f"Loading from UC failed ({e}), loading from run URI instead...")
    loaded_model = mlflow.pyfunc.load_model(model_uri)
    prediction = loaded_model.predict(input_example)
    display(prediction)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-78adeb80-dab3-4670-ba5e-b0fadd786ddd/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


,answer
0,The average customer value for the VIP segment...


Trace(trace_id=tr-b279d993e65d29cf32408e1d4ba0b6bb)



---
## Akt 5: Wdrożenie — Model Serving, batch `ai_query`, Databricks App, inference table (\~30 min)

Wdrażamy agenta jako **Model Serving endpoint** i tworzymy **Databricks App** z Gradio — całość programatycznie, prosto z tego notebooka.

**Architektura:**

Użytkownik (przeglądarka) → Databricks App (Gradio) → Model Serving endpoint (`@champion`) → Agent → UC Functions → Delta table
                                                                    └─► AI Gateway inference table → monitoring (WS2 Cz. 4 §7)

**Kroki:**
1. Tworzymy Model Serving endpoint z wersji **`@champion`** modelu UC; włączamy **inference table** (AI Gateway) i usage tracking
2. Testujemy endpoint — przez Databricks SDK i przez `mlflow.deployments`
3. **Batch inference w SQL**: `ai_query()` na **własnym** endpoincie — 5 pytań analityków → tabela Delta
4. Zapisujemy `app.py` + `app.yaml` + `requirements.txt` do workspace i wdrażamy **Databricks App** przez SDK
5. Zaglądamy do **inference table** — surowe request/response agenta; to wejście do monitora Time Series z WS2

> *"Cała aplikacja — od endpointu po UI — powstaje z kodu w notebooku. Infrastructure as Code."*

In [0]:
%python


# Akt 5 | Tworzenie Model Serving endpoint z wersji @champion + AI Gateway (inference table, usage tracking)
import requests
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

import mlflow
mlflow.set_registry_uri("databricks-uc")
_client = mlflow.MlflowClient()

# ai_query / Model Serving celują w NUMER wersji — rozwiązujemy alias @champion do wersji (jak w kursie deploy & monitor)
_champion = _client.get_model_version_by_alias(name=uc_model_name, alias="champion")
_champion_version = str(_champion.version)

w = WorkspaceClient()
endpoint_name = "workshop-retail-agent"
inference_table_prefix = "retail_agent_inference"
inference_payload_table = f"{catalog}.{schema}.{inference_table_prefix}_payload"

print(f"Tworzenie endpointu '{endpoint_name}' z modelem {uc_model_name}@champion (= v{_champion_version})...")
print("To może zająć ~10 minut. W międzyczasie omówimy architekturę aplikacji.\n")

served_entity = ServedEntityInput(
    name="retail-agent-champion",
    entity_name=uc_model_name,
    entity_version=_champion_version,
    scale_to_zero_enabled=True,
    workload_size="Small",
)

try:
    w.serving_endpoints.create(name=endpoint_name, config=EndpointCoreConfigInput(name=endpoint_name, served_entities=[served_entity]))
    print(f"Endpoint '{endpoint_name}' — tworzenie uruchomione.")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Endpoint '{endpoint_name}' już istnieje — aktualizuję do @champion v{_champion_version}...")
        w.serving_endpoints.wait_get_serving_endpoint_not_updating(endpoint_name)
        w.serving_endpoints.update_config(name=endpoint_name, served_entities=[served_entity])
        print(f"Endpoint '{endpoint_name}' zaktualizowany.")
    else:
        raise

# AI Gateway: inference table (surowe request/response w UC) + usage tracking. PUT jest idempotentny.
_headers = w.config.authenticate()
_gateway_cfg = {
    "usage_tracking_config": {"enabled": True},
    "inference_table_config": {"enabled": True, "catalog_name": catalog, "schema_name": schema, "table_name_prefix": inference_table_prefix},
}
_resp = requests.put(f"{w.config.host.rstrip('/')}/api/2.0/serving-endpoints/{endpoint_name}/ai-gateway", headers=_headers, json=_gateway_cfg)
if _resp.ok:
    print(f"AI Gateway: inference table włączona → {inference_payload_table} (payloady pojawiają się z opóźnieniem kilku–kilkunastu minut)")
else:
    print(f"[INFO] Nie udało się włączyć AI Gateway ({_resp.status_code}): {_resp.text[:200]}")
    print("       Alternatywa w UI: Serving → workshop-retail-agent → Edit AI Gateway → Inference tables → sandbox / testy / retail_agent_inference")

print(f"\nSprawdź status: Serving → Endpoints → {endpoint_name}")

Tworzenie endpointu 'workshop-retail-agent' z modelem workspace.default.retail_customer_agent@champion (= v1)...
To może zająć ~10 minut. W międzyczasie omówimy architekturę aplikacji.

Endpoint 'workshop-retail-agent' — tworzenie uruchomione.
[INFO] Nie udało się włączyć AI Gateway (409): {"error_code":"RESOURCE_CONFLICT","message":"Endpoint served entities are currently being updated. Please try again after the current update is no longer in progress.","details":[{"@type":"type.google
       Alternatywa w UI: Serving → workshop-retail-agent → Edit AI Gateway → Inference tables → sandbox / testy / retail_agent_inference

Sprawdź status: Serving → Endpoints → workshop-retail-agent


In [0]:
%python


# Akt 5 | Test endpointu (uruchom gdy endpoint jest READY)
print(f"Czekam na gotowość endpointu '{endpoint_name}'...")
endpoint_info = w.serving_endpoints.wait_get_serving_endpoint_not_updating(endpoint_name)
print(f"Status: {endpoint_info.state.ready}\n")

# Wyślij testowe zapytanie — to samo, którym testowaliśmy agenta lokalnie w Akcie 3
test_question = f"What is the average customer value for the VIP segment and the profile of customer {sample_customer_id}?"
print(f"Pytanie: {test_question}\n")

response = w.serving_endpoints.query(
    name=endpoint_name,
    dataframe_records=[{"prompt": test_question}]
)

print("--- Odpowiedź z Model Serving (Databricks SDK) ---")
try:
    print(response.predictions[0]["answer"])
except (KeyError, IndexError, TypeError):
    print(response.as_dict())

Czekam na gotowość endpointu 'workshop-retail-agent'...
Status: EndpointStateReady.READY

Pytanie: What is the average customer value for the VIP segment and the profile of customer 1668?

--- Odpowiedź z Model Serving (Databricks SDK) ---
The average customer value for the VIP segment is $1038.72. The profile of customer 1668 is as follows: 
Name: NGUYEN, LINH THI MY
Location: LUNENBURG, VT
Loyalty segment: New (0)
Units purchased: 5
Total spend: $0.0
Avg item value: $0.0
Orders: 0 
RFM Recency: 999 days
RFM Frequency: 0


In [0]:

%python
# Akt 5 | Alternatywny klient: MLflow Deployments — ten sam payload (dataframe_records), inny SDK.
# Przydatny w kodzie, który ma być przenośny między MLflow a Databricks (np. w testach CI).
import mlflow.deployments

deploy_client = mlflow.deployments.get_deploy_client("databricks")
deployments_response = deploy_client.predict(
    endpoint=endpoint_name,
    inputs={"dataframe_records": [{"prompt": f"Compare the average customer value of the Regular (2) and Occasional (1) segments."}]},
)
print("--- Odpowiedź z Model Serving (mlflow.deployments) ---")
try:
    print(deployments_response["predictions"][0]["answer"])
except (KeyError, IndexError, TypeError):
    print(deployments_response)

--- Odpowiedź z Model Serving (mlflow.deployments) ---
The average customer value of the Regular (2) segment is $96.96, while the average customer value of the Occasional (1) segment is $49.77. This indicates that customers in the Regular segment have a significantly higher average spend than those in the Occasional segment, with a difference of $47.19.




### Batch inference: `ai_query()` na **naszym** endpoincie

W WS1 `ai_query()` wołało gotowy foundation model. Ta sama funkcja SQL działa na **dowolnym endpoincie Model Serving** —
także na naszym agencie. Zamiast pętli w Pythonie: tabela pytań analityków → `ai_query` → tabela odpowiedzi w Delta.
To wzorzec z kursu *Deploying and Monitoring Agent Applications*: **ten sam model obsługuje ruch online (App) i batch (SQL)**.

| Element | Wartość |
| --- | --- |
| `request` | `named_struct('prompt', question)` — kolumna `prompt` to sygnatura wejścia naszego pyfunc |
| `returnType` | `STRUCT<answer:STRING>` — model zwraca DataFrame z kolumną `answer` |
| Wynik | `workspace.default.retail_agent_batch_answers` — zwykła tabela Delta (dashboard, Genie, monitoring) |

> Endpoint ze *scale to zero* może potrzebować ~1 min na „wybudzenie” przy pierwszym wywołaniu. Jeżeli składnia
> `ai_query` dla custom modeli nie jest dostępna w Twoim runtime, komórka użyje pętli przez SDK — wynik jest identyczny.

In [0]:

%python
# Akt 5 | Batch: 5 pytań analityków → ai_query('workshop-retail-agent') → tabela odpowiedzi
questions_table = f"{catalog}.{schema}.retail_agent_batch_questions"
answers_table = f"{catalog}.{schema}.retail_agent_batch_answers"

analyst_questions = [
    (1, "What is the average customer value for the VIP segment?"),
    (2, "Compare the average customer value of Regular (2) and Occasional (1) segments."),
    (3, f"Give me the profile of customer {sample_customer_id}."),
    (4, "Which loyalty segment has the highest average spend and what does that suggest for retention?"),
    (5, f"What is the tax_id of customer {sample_customer_id}?"),   # PII probe — agent powinien odmówić także w batchu
]
spark.createDataFrame(analyst_questions, ["question_id", "question"]).write.mode("overwrite").saveAsTable(questions_table)
print(f"Pytania: {questions_table} ({len(analyst_questions)} wierszy)")

try:
    spark.sql(f"""
        CREATE OR REPLACE TABLE {answers_table} AS
        SELECT
          question_id,
          question,
          ai_query(
            '{endpoint_name}',
            named_struct('prompt', question),
            returnType => 'STRUCT<answer:STRING>'
          ).answer AS answer,
          current_timestamp() AS answered_at
        FROM {questions_table}
    """)
    method = "ai_query (SQL)"
except Exception as e:
    print(f"[INFO] ai_query na custom endpoincie niedostępne w tym runtime ({type(e).__name__}) — używam pętli przez SDK.")
    rows = []
    for qid, q in analyst_questions:
        r = w.serving_endpoints.query(name=endpoint_name, dataframe_records=[{"prompt": q}])
        rows.append((qid, q, r.predictions[0]["answer"]))
    (spark.createDataFrame(rows, ["question_id", "question", "answer"])
          .withColumn("answered_at", F.current_timestamp())
          .write.mode("overwrite").saveAsTable(answers_table))
    method = "SDK loop"

print(f"Odpowiedzi ({method}): {answers_table}")
display(spark.table(answers_table).orderBy("question_id"))
print("\n💡 Pytanie 5 (tax_id) powinno skończyć się odmową — guardrails działają identycznie w App i w batchu, bo to TEN SAM model.")

Pytania: workspace.default.retail_agent_batch_questions (5 wierszy)
Odpowiedzi (ai_query (SQL)): workspace.default.retail_agent_batch_answers


question_id,question,answer,answered_at
1,What is the average customer value for the VIP segment?,"The average customer value for the VIP segment is $1038.72. This suggests that, on average, customers in the VIP loyalty segment have spent around $1038.72, indicating a high level of engagement and loyalty to the brand.",2026-09-08T13:49:43.861Z
2,Compare the average customer value of Regular (2) and Occasional (1) segments.,"The average customer value of the Regular (2) segment is $96.96, while the average customer value of the Occasional (1) segment is $49.77. This indicates that customers in the Regular segment have a significantly higher average spend than those in the Occasional segment, with the Regular segment having an average customer value nearly twice that of the Occasional segment.",2026-09-08T13:49:43.861Z
3,Give me the profile of customer 1668.,"The customer profile for customer 1668 is as follows: Customer ID: 1668 Name: NGUYEN, LINH THI MY Location: LUNENBURG, VT Loyalty segment: New (0) Units purchased: 5 Total spend: $0.0 Avg item value: $0.0 Orders: 0 (promo: 0, 0.0%) RFM Recency: 999 days RFM Frequency: 0 This suggests that customer 1668 is a new customer who has not made any orders, but has purchased 5 units. The total spend and average item value are $0.0, indicating that the units purchased may not have been part of a paid order. The RFM recency is 999 days, which is the maximum value, indicating that the customer has not made an order in a long time, if ever. The RFM frequency is 0, which also indicates that the customer has not made any orders.",2026-09-08T13:49:43.861Z
4,Which loyalty segment has the highest average spend and what does that suggest for retention?,"The VIP loyalty segment (segment 3) has the highest average spend, with an average of $1038.72. This suggests that retaining VIP customers is crucial for the business, as they generate significantly more revenue than customers in other loyalty segments. Focusing on personalized marketing efforts, loyalty rewards, and exceptional customer service may help to retain these high-value customers and encourage continued loyalty.",2026-09-08T13:49:43.861Z
5,What is the tax_id of customer 1668?,"I'm not able to provide the tax_id of customer 1668 as it is considered protected personally identifiable information (PII). Instead, I can offer to provide anonymized information about the customer or help with other retail analytics questions. Would you like to know more about the customer's loyalty segment, order history, or RFM metrics?",2026-09-08T13:49:43.861Z



💡 Pytanie 5 (tax_id) powinno skończyć się odmową — guardrails działają identycznie w App i w batchu, bo to TEN SAM model.


In [0]:
%python

# Akt 5 | Przygotowanie kodu źródłowego Databricks App
from databricks.sdk.service.workspace import ImportFormat

app_name = "workshop-retail-agent"
app_source_path = f"/Workspace/Users/{username}/apps/{app_name}"

# --- app.py: Gradio chatbox wywołujący Model Serving endpoint ---
app_py = f'''import gradio as gr
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
ENDPOINT = "{endpoint_name}"


def chat(message, history):
    """Wyślij pytanie do agenta retail przez Model Serving."""
    response = w.serving_endpoints.query(
        name=ENDPOINT,
        dataframe_records=[{{"prompt": message}}],
    )
    try:
        return response.predictions[0]["answer"]
    except (KeyError, IndexError, TypeError):
        return str(response.as_dict())


demo = gr.ChatInterface(
    fn=chat,
    title="Retail Customer Agent — Workshop 4",
    description="Rozmawiaj z AI agentem o danych klientów B2B",
    examples=[
        "What is the average customer value for VIP segment?",
        "Tell me about customer {sample_customer_id}",
        "Compare Regular vs Occasional segments",
    ],
)

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=8000)
'''

# --- app.yaml: konfiguracja Databricks App ---
app_yaml = """command:
  - python
  - app.py
env:
  - name: DATABRICKS_HOST
    valueFrom: databricks-sdk
  - name: DATABRICKS_TOKEN
    valueFrom: databricks-sdk
"""

# --- requirements.txt: zależności ---
requirements_txt = """gradio>=4.0
databricks-sdk>=0.30.0
"""

# Zapis plików do workspace
w.workspace.mkdirs(app_source_path)
for fname, content in [("app.py", app_py), ("app.yaml", app_yaml), ("requirements.txt", requirements_txt)]:
    w.workspace.upload(
        path=f"{app_source_path}/{fname}",
        content=content.encode("utf-8"),
        overwrite=True,
        format=ImportFormat.AUTO,
    )
    print(f"  Zapisano: {app_source_path}/{fname}")

print(f"\nKod źródłowy aplikacji gotowy w: {app_source_path}")

  Zapisano: /Workspace/Users/katarzyna.palach@cloudsonmars.com/apps/workshop-retail-agent/app.py
  Zapisano: /Workspace/Users/katarzyna.palach@cloudsonmars.com/apps/workshop-retail-agent/app.yaml
  Zapisano: /Workspace/Users/katarzyna.palach@cloudsonmars.com/apps/workshop-retail-agent/requirements.txt

Kod źródłowy aplikacji gotowy w: /Workspace/Users/katarzyna.palach@cloudsonmars.com/apps/workshop-retail-agent


In [0]:
%python

# Akt 5 | Tworzenie i deployment Databricks App
from databricks.sdk.service.apps import App, AppDeployment

print(f"Tworzenie aplikacji '{app_name}'...\n")

try:
    app = w.apps.create_and_wait(
        app=App(name=app_name, description="Retail Customer Agent — Workshop 4"),
    )
    print(f"Aplikacja '{app_name}' utworzona.")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Aplikacja '{app_name}' już istnieje — kontynuuję deployment.")
    else:
        raise

# Ensure app compute is active before deploying
import time as _time
app_check = w.apps.get(name=app_name)
compute_state = str(getattr(getattr(app_check, 'compute_status', None), 'state', 'UNKNOWN'))
if 'ACTIVE' not in compute_state:
    print(f"App compute w stanie {compute_state} \u2014 uruchamiam...")
    w.apps.start(name=app_name)
    for _ in range(30):
        _time.sleep(10)
        app_check = w.apps.get(name=app_name)
        compute_state = str(getattr(getattr(app_check, 'compute_status', None), 'state', 'UNKNOWN'))
        if 'ACTIVE' in compute_state:
            break
    print(f"App compute state: {compute_state}")

print(f"Wdrażanie z: {app_source_path}...\n")

deployment = w.apps.deploy_and_wait(
    app_name=app_name,
    app_deployment=AppDeployment(source_code_path=app_source_path),
)
print(f"Status deployment: {deployment.status.state}")

# Pobierz URL aplikacji i nadaj uprawnienia
app_info = w.apps.get(name=app_name)

# App SP potrzebuje CAN_QUERY na endpoint żeby móc wywoływać agenta
import requests
host = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
ep = w.serving_endpoints.get(name=endpoint_name)
resp = requests.patch(
    f"https://{host}/api/2.0/permissions/serving-endpoints/{ep.id}",
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={"access_control_list": [{"service_principal_name": app_info.service_principal_client_id, "permission_level": "CAN_QUERY"}]},
)
if resp.status_code == 200:
    print(f"Nadano CAN_QUERY na endpoint dla app SP '{app_info.service_principal_client_id}'")
else:
    print(f"Uwaga: nie udało się nadać uprawnień ({resp.status_code}): {resp.text}")

if app_info.url:
    print(f"\nAplikacja dostępna: {app_info.url}")
    print("Otwórz w przeglądarce i zadaj pytanie agentowi!")
else:
    print(f"\nAplikacja uruchomiona. Sprawdź URL w: Apps → {app_name}")

Tworzenie aplikacji 'workshop-retail-agent'...

Aplikacja 'workshop-retail-agent' już istnieje — kontynuuję deployment.
Wdrażanie z: /Workspace/Users/katarzyna.palach@cloudsonmars.com/apps/workshop-retail-agent...

Status deployment: AppDeploymentState.SUCCEEDED
Nadano CAN_QUERY na endpoint dla app SP '6102c591-4e7f-4fb6-84d6-1d3cd9381f87'

Aplikacja dostępna: https://workshop-retail-agent-7405615837166522.2.azure.databricksapps.com
Otwórz w przeglądarce i zadaj pytanie agentowi!




### Inference table: co dokładnie ludzie pytają agenta — i co on odpowiada

Endpoint z włączonym AI Gateway zapisuje **każde** wywołanie (z App, z SDK, z `ai_query`) do tabeli
`workspace.default.retail_agent_inference_payload`: `request` (JSON z `dataframe_records[0].prompt`), `response`
(JSON z `predictions[0].answer`), czas, status, tożsamość wywołującego. Payloady docierają **z opóźnieniem**
(kilka–kilkanaście minut), więc na warsztacie tabela może być jeszcze pusta — komórka to obsłuży.

**To jest domknięcie historii z WS2:** tam (Cz. 4 §7) zbudowaliśmy pipeline *payload → metryki (toxicity, readability,
refusal) → monitor Time Series*. Tu wystarczy podmienić tabelę źródłową i ścieżki JSON — reszta działa bez zmian.

In [0]:

%python
# Akt 5 | Inference table (AI Gateway): request/response → tekst pytania i odpowiedzi
from pyspark.sql import functions as F

if not spark.catalog.tableExists(inference_payload_table):
    print(f"ℹ️  {inference_payload_table} jeszcze nie istnieje — payloady AI Gateway pojawiają się kilka–kilkanaście minut po pierwszych wywołaniach.")
    print("   Sprawdź: Serving → workshop-retail-agent → AI Gateway → Inference table. Uruchom tę komórkę ponownie później.")
else:
    payload_df = spark.table(inference_payload_table)
    cols = payload_df.columns
    # Table may exist but be schema-only (no payload columns yet) while AI Gateway is provisioning
    if "request" not in cols or "response" not in cols:
        print(f"ℹ️  {inference_payload_table} istnieje, ale payloady jeszcze nie dotarły (kolumny: {cols}).")
        print("   AI Gateway dostarcza dane z opóźnieniem kilku–kilkunastu minut. Uruchom tę komórkę ponownie później.")
    else:
      ts_col = F.col("request_time") if "request_time" in cols else F.expr("timestamp_millis(timestamp_ms)")
      unpacked = (
          payload_df
          .withColumn("timestamp", ts_col)
          .withColumn("input_text", F.get_json_object(F.col("request"), "$.dataframe_records[0].prompt"))
          .withColumn("output_text", F.get_json_object(F.col("response"), "$.predictions[0].answer"))
          .select("databricks_request_id", "timestamp", "status_code", "input_text", "output_text")
          .orderBy(F.desc("timestamp"))
      )
      n = unpacked.count()
      print(f"📡 {inference_payload_table}: {n} wywołań agenta")
      if n:
          display(unpacked.limit(10))
          print("\n➡️  Następny krok (WS2 Cz. 4 §7): PAYLOAD_TABLE = inference_payload_table, ścieżki JSON jak wyżej,")
          print("    metryki toxicity/readability/is_refusal → monitor Time Series 5 min na odpowiedziach AGENTA.")
      else:
          print("   Tabela istnieje, ale jest jeszcze pusta — poczekaj na dostarczenie payloadów.")

📡 workspace.default.retail_agent_inference_payload: 7 wywołań agenta


databricks_request_id,timestamp,status_code,input_text,output_text
f7b1b6e1-2714-4716-a83e-00f0c168d6a1,2026-09-08T13:50:02.513Z,200,What is the tax_id of customer 1668?,"I'm not able to provide the tax_id of customer 1668 as it is considered protected personally identifiable information (PII). Instead, I can offer to provide anonymized information about the customer or help with other retail analytics questions. Would you like to know more about the customer's loyalty segment, order history, or RFM metrics?"
68510e9e-3c49-4e7f-9acd-56c74e4d2d58,2026-09-08T13:50:01.482Z,200,Which loyalty segment has the highest average spend and what does that suggest for retention?,"The VIP loyalty segment (segment 3) has the highest average spend, with an average of $1038.72. This suggests that retaining VIP customers is crucial for the business, as they generate significantly more revenue than customers in other loyalty segments. Focusing on personalized marketing efforts, loyalty rewards, and exceptional customer service may help to retain these high-value customers and encourage continued loyalty."
55c9c15a-9344-4d38-a0fb-706e86892f1e,2026-09-08T13:49:57.215Z,200,Give me the profile of customer 1668.,"The customer profile for customer 1668 is as follows: Customer ID: 1668 Name: NGUYEN, LINH THI MY Location: LUNENBURG, VT Loyalty segment: New (0) Units purchased: 5 Total spend: $0.0 Avg item value: $0.0 Orders: 0 (promo: 0, 0.0%) RFM Recency: 999 days RFM Frequency: 0 This suggests that customer 1668 is a new customer who has not made any orders, but has purchased 5 units. The total spend and average item value are $0.0, indicating that the units purchased may not have been part of a paid order. The RFM recency is 999 days, which is the maximum value, indicating that the customer has not made an order in a long time, if ever. The RFM frequency is 0, which also indicates that the customer has not made any orders."
81c884a3-0371-4cee-9218-c30a8f6de11b,2026-09-08T13:49:52.658Z,200,Compare the average customer value of Regular (2) and Occasional (1) segments.,"The average customer value of the Regular (2) segment is $96.96, while the average customer value of the Occasional (1) segment is $49.77. This indicates that customers in the Regular segment have a significantly higher average spend than those in the Occasional segment, with the Regular segment having an average customer value nearly twice that of the Occasional segment."
b1158803-0043-4907-bd19-0f51aa5b294a,2026-09-08T13:49:48.295Z,200,What is the average customer value for the VIP segment?,"The average customer value for the VIP segment is $1038.72. This suggests that, on average, customers in the VIP loyalty segment have spent around $1038.72, indicating a high level of engagement and loyalty to the brand."
bec832ca-9040-483d-8a25-b5e08f7f6fea,2026-09-08T13:49:34.877Z,200,Compare the average customer value of the Regular (2) and Occasional (1) segments.,"The average customer value of the Regular (2) segment is $96.96, while the average customer value of the Occasional (1) segment is $49.77. This indicates that customers in the Regular segment have a significantly higher average spend than those in the Occasional segment, with a difference of $47.19."
d7c2f856-952b-46e9-867d-33acf3eb3304,2026-09-08T13:49:30.265Z,200,What is the average customer value for the VIP segment and the profile of customer 1668?,"The average customer value for the VIP segment is $1038.72. The profile of customer 1668 is as follows: Name: NGUYEN, LINH THI MY Location: LUNENBURG, VT Loyalty segment: New (0) Units purchased: 5 Total spend: $0.0 Avg item value: $0.0 Orders: 0 RFM Recency: 999 days RFM Frequency: 0"



➡️  Następny krok (WS2 Cz. 4 §7): PAYLOAD_TABLE = inference_payload_table, ścieżki JSON jak wyżej,
    metryki toxicity/readability/is_refusal → monitor Time Series 5 min na odpowiedziach AGENTA.




### Databricks App

Endpoint serwuje agenta przez REST API. Interfejs webowy to **Databricks App** z Gradio —
chatbox, przez który użytkownicy rozmawiają z agentem o danych klientów B2B.

Aplikacja została utworzona i wdrożona programatycznie z tego notebooka —
kod źródłowy leży w workspace, a SDK Databricks obsłużył tworzenie i deployment.

---

### Podsumowanie Warsztatu 4

Zbudowaliśmy kompletny pipeline od surowych danych do produkcyjnej aplikacji:

`Gold Table (WS1) → Guardrails (WS2) → RAG (WS3) → UC Functions (bez PII) → Agent + Guardrails → MCP Google Drive (Docs/Sheets/Slides) → MLflow Tracing → UC Model @champion → Serving + inference table → batch ai_query → App → monitoring (WS2 §7)`

| Akt | Co zbudowaliśmy |
| --- | --- |
| 1 | Konfiguracja, przegląd Gold Table z WS1 |
| 2 | 3 UC Functions (avg value, profil bez PII, formatowanie) + test payloadem + AI Playground |
| 3 | Agent LangChain z guardrails + MLflow Tracing (Workspace i Unity Catalog) |
| 3b | **MCP Google Drive** — agent rozszerzony o 13 narzędzi Google (create/read/edit Docs, Sheets, Slides) przez Model Context Protocol |
| 4 | Rejestracja modelu w UC + alias `@champion` |
| 5 | Model Serving z inference table, batch `ai_query`, Databricks App (Gradio), monitoring |

**Pytania do dyskusji:**
1. Jak połączyć Knowledge Assistant (WS3) z tym agentem? (Odp: Supervisor Agent)
2. Jak rozszerzyć guardrails? (Odp: Databricks safety filter z WS2 + row filter — system prompt + safety filter)
3. Jak monitorować jakość odpowiedzi agenta? (Odp: inference table z Aktu 5 → pipeline z WS2 Cz. 4 §7: toxicity/readability/refusal → monitor Time Series; scorery `mlflow.genai` na trace'ach)
4. Co zmienić żeby app obsługiwała wielu użytkowników? (Odp: autoscaling endpoint + app replicas)
5. Jak wypuścić nową wersję agenta bez zmiany App? (Odp: zarejestruj v2 → przepnij alias `@champion` → `update_config` endpointu; App wskazuje endpoint, nie wersję)
6. Jak dodać kolejne źródła MCP? (Odp: `DatabricksMultiServerMCPClient` przyjmuje listę serwerów — dodaj Genie, AI Search, Slack itp. obok Google Drive)

In [0]:
%python
# Akt 5 | Cleanup — uruchom po warsztacie żeby zatrzymać koszty
# Odkomentuj i uruchom żeby usunąć zasoby:

# # Zatrzymaj app
# try:
#     w.apps.stop(name=app_name)
#     print(f"App '{app_name}' zatrzymana.")
# except Exception as e:
#     print(f"App stop: {e}")

# # Usuń serving endpoint
# try:
#     w.serving_endpoints.delete(name=endpoint_name)
#     print(f"Endpoint '{endpoint_name}' usunięty.")
# except Exception as e:
#     print(f"Endpoint delete: {e}")

print("⚠️ Odkomentuj linie powyżej i uruchom żeby zatrzymać koszty po warsztacie.")
print(f"  App: {app_name}")
print(f"  Endpoint: {endpoint_name}")
print(f"  Model: {uc_model_name}")